# Loading the dataset

In [38]:
import pandas as pd
gen_data = pd.read_csv("../data/raw/fp-historical-wildfire-data-2006-2025.csv", low_memory=False)

In [39]:
# select only the columns we need for our analysis()
attributes_of_interest = [
    "YEAR",
    "FIRE_NUMBER",
    "SIZE_CLASS",
    "FIRE_ORIGIN",
    "DETECTION_AGENT_TYPE",
    "REPORTED_DATE",
    "DISPATCHED_RESOURCE",
    "DISPATCH_DATE",
    "ASSESSMENT_DATETIME",
    "ASSESSMENT_HECTARES",
    "FIRE_SPREAD_RATE",
    "FIRE_TYPE",
    "FIRE_POSITION_ON_SLOPE",
    "WEATHER_CONDITIONS_OVER_FIRE",
    "TEMPERATURE",
    "RELATIVE_HUMIDITY",
    "WIND_DIRECTION",
    "WIND_SPEED",
    "FUEL_TYPE"
]

gen_data = gen_data[attributes_of_interest]

In [40]:
# Extract forest area from the wildfire number
gen_data["FOREST_AREA"] = (
    gen_data["FIRE_NUMBER"].astype(str).str[0]
)


In [41]:
# missing values in the dataset
gen_data.isnull().sum()
#gen_data["FIRE_ORIGIN"].head(20)

YEAR                               0
FIRE_NUMBER                        0
SIZE_CLASS                         0
FIRE_ORIGIN                       12
DETECTION_AGENT_TYPE               0
REPORTED_DATE                      0
DISPATCHED_RESOURCE               15
DISPATCH_DATE                     15
ASSESSMENT_DATETIME                0
ASSESSMENT_HECTARES                0
FIRE_SPREAD_RATE                2858
FIRE_TYPE                       2650
FIRE_POSITION_ON_SLOPE          2919
WEATHER_CONDITIONS_OVER_FIRE    2925
TEMPERATURE                     2927
RELATIVE_HUMIDITY               2935
WIND_DIRECTION                  2935
WIND_SPEED                      2935
FUEL_TYPE                       7760
FOREST_AREA                        0
dtype: int64

### Handle categorical missing data

In [42]:
categorical_columns = [
    "FIRE_ORIGIN",
    "DISPATCHED_RESOURCE",
    "FIRE_TYPE",
    "FIRE_POSITION_ON_SLOPE",
    "WEATHER_CONDITIONS_OVER_FIRE",
    "WIND_DIRECTION",
    "FUEL_TYPE"
]

gen_data[categorical_columns] = gen_data[categorical_columns].fillna("Unknown")


In [43]:
gen_data.dtypes

YEAR                              int64
FIRE_NUMBER                      object
SIZE_CLASS                       object
FIRE_ORIGIN                      object
DETECTION_AGENT_TYPE             object
REPORTED_DATE                    object
DISPATCHED_RESOURCE              object
DISPATCH_DATE                    object
ASSESSMENT_DATETIME              object
ASSESSMENT_HECTARES             float64
FIRE_SPREAD_RATE                float64
FIRE_TYPE                        object
FIRE_POSITION_ON_SLOPE           object
WEATHER_CONDITIONS_OVER_FIRE     object
TEMPERATURE                     float64
RELATIVE_HUMIDITY               float64
WIND_DIRECTION                   object
WIND_SPEED                      float64
FUEL_TYPE                        object
FOREST_AREA                      object
dtype: object

In [44]:
gen_data.describe()

,YEAR,ASSESSMENT_HECTARES,FIRE_SPREAD_RATE,TEMPERATURE,RELATIVE_HUMIDITY,WIND_SPEED
count,27828.000000,27828.000000,24970.000000,24901.000000,24893.000000,24893.000000
mean,2014.709070,2.429860,0.909880,17.878888,45.203270,8.922830
std,5.800447,58.704752,2.602409,7.784500,18.675777,8.418009
min,2006.000000,0.010000,-1.000000,-39.000000,0.000000,0.000000
25%,2010.000000,0.010000,0.000000,14.000000,31.000000,3.000000
50%,2014.000000,0.010000,0.000000,19.000000,40.000000,7.000000
75%,2020.000000,0.250000,1.000000,23.000000,56.000000,12.000000
max,2025.000000,6019.000000,100.000000,60.000000,100.000000,90.000000


#### Splitting the data into Training (70%), Validation(15%) and Test(15%)

In [45]:
# Splitting the data into Training (70%), Validation(15%) and Test(15%)

from sklearn.model_selection import train_test_split

# Step 1: 70% training, 30% remaining for validation and test
train_data, remaining_data = train_test_split(
    gen_data, 
    test_size=0.3, 
    random_state=42,
    stratify=gen_data["SIZE_CLASS"]
)

# Step 2: Split the remaining 30% into validation and test (15% each)
validation_data, test_data = train_test_split(
    remaining_data,
    test_size=0.5,
    random_state=42,
    stratify=remaining_data["SIZE_CLASS"]
)

 

#### Handle missing value by using the mean

In [46]:
from sklearn.impute import SimpleImputer

numeric_columns = train_data.select_dtypes(include="number").columns
imputer = SimpleImputer(strategy="median")

train_data[numeric_columns] = imputer.fit_transform(train_data[numeric_columns])

validation_data[numeric_columns] = imputer.transform(validation_data[numeric_columns])

test_data[numeric_columns] = imputer.transform(test_data[numeric_columns])

#### Now write data to SCV files

In [47]:
# Now write data to CSV files
train_data.to_csv("../data/processed/train_data.csv", index=False)
validation_data.to_csv("../data/processed/validation_data.csv", index=False)
test_data.to_csv("../data/processed/test_data.csv", index=False)

In [48]:
train_data.head(10)

,YEAR,FIRE_NUMBER,SIZE_CLASS,FIRE_ORIGIN,DETECTION_AGENT_TYPE,REPORTED_DATE,DISPATCHED_RESOURCE,DISPATCH_DATE,ASSESSMENT_DATETIME,ASSESSMENT_HECTARES,FIRE_SPREAD_RATE,FIRE_TYPE,FIRE_POSITION_ON_SLOPE,WEATHER_CONDITIONS_OVER_FIRE,TEMPERATURE,RELATIVE_HUMIDITY,WIND_DIRECTION,WIND_SPEED,FUEL_TYPE,FOREST_AREA
509,2006.0,MWF103,E,Provincial Land,LKT,2006-06-29 15:40,RAP,2006-06-29 15:41,2006-06-29 15:51,1.00,5.0,Crown,Flat,Clear,22.0,24.0,SW,25.0,C2,M
16953,2016.0,HWF016,B,Indian Reservation,UNP,2016-04-22 20:55,HAC,2016-04-22 21:15,2016-04-22 23:00,0.40,3.0,Surface,Flat,Clear,0.0,45.0,SE,15.0,O1a,H
19091,2018.0,HWF136,C,Provincial Land,UNP,2018-06-22 14:03,FPD Staff,2018-06-22 14:03,2018-06-22 14:04,3.00,3.0,Crown,Flat,Cloudy,20.0,75.0,CLM,0.0,C1,H
22207,2021.0,WWF097,B,Provincial Land,UNP,2021-07-22 15:17,HAC,2021-07-22 15:34,2021-07-22 17:43,0.02,0.0,Ground,Flat,CB Dry,17.0,35.0,NW,15.0,S1,W
19347,2018.0,SWF062,B,Provincial Land,LKT,2018-06-21 16:20,RAP,2018-06-21 16:27,2018-06-21 16:20,1.50,1.0,Surface,Middle 1/3,CB Wet,23.0,75.0,CLM,0.0,M2,S
12649,2013.0,EWF061,A,Provincial Land,LKT,2013-08-03 10:39,FPD Staff,2013-08-03 10:53,2013-08-03 12:00,0.01,0.0,Surface,Flat,CB Dry,21.0,35.0,NW,5.0,Unknown,E
6328,2009.0,RWF130,A,Provincial Land,GRP,2009-08-03 13:35,FPD Staff,2009-08-03 13:38,2009-08-03 13:38,0.01,0.0,Surface,Flat,Cloudy,13.0,70.0,E,2.0,Unknown,R
9977,2012.0,LWF086,B,Provincial Land,LKT,2012-05-31 11:25,HAC,2012-05-31 11:25,2012-05-31 11:34,0.20,1.0,Surface,Bottom,Cloudy,22.0,35.0,SE,5.0,O1a,L
17246,2016.0,EWF024,B,Provincial Land,UNP,2016-04-18 12:32,UNIT,2016-04-18 12:41,2016-04-18 13:28,0.10,2.0,Surface,Flat,Clear,23.0,22.0,SW,14.0,O1a,E
3906,2008.0,CWF016,A,Provincial Land,UNP,2008-05-06 10:05,FPD Staff,2008-05-06 10:20,2008-05-06 11:10,0.01,0.0,Surface,Flat,Cloudy,10.0,60.0,NW,5.0,Unknown,C
